# M3L2 E03 - Tools con @tool y API del dolar

## Objetivo

Vas a convertir funciones Python en tools, ver `tool_calls` y consultar DolarAPI desde una herramienta.

## Tool bien disenada

| Parte | Regla |
|---|---|
| Nombre | Verbo claro |
| Input | Parametros simples y tipados |
| Output | Resultado predecible |
| Docstring | Explica cuando usarla |
| Error | Mensaje claro si falla |

## Diagrama

```text
Pregunta -> LLM con tools -> tool_call -> Python ejecuta tool -> observacion
```

In [ ]:
# !pip install langchain langchain-openai requests

In [ ]:
import os
import getpass
from langchain_openai import ChatOpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OpenAI API key: ")


def obtener_modelo(temperature: float = 0.2):
    return ChatOpenAI(model="gpt-4o-mini", temperature=temperature)


print("API key cargada en la variable de entorno OPENAI_API_KEY")

In [ ]:
import requests
from langchain_core.tools import tool

llm = obtener_modelo()

In [ ]:
@tool
def calculadora_magica(expresion: str) -> str:
    """Evalua una expresion matematica simple enviada como texto."""
    return str(eval(expresion))

print(calculadora_magica.name)
print(calculadora_magica.description)
print(calculadora_magica.invoke({"expresion": "2 + 3 * 4"}))

In [ ]:
llm_con_herramientas = llm.bind_tools([calculadora_magica])
respuesta = llm_con_herramientas.invoke("Cuanto es 21 * 5? Usa una herramienta si hace falta.")
print(respuesta)
print("Tool calls:", respuesta.tool_calls)

In [ ]:
@tool
def consultar_dolar_blue() -> str:
    """Consulta la cotizacion actual del dolar blue en Argentina usando DolarAPI."""
    url = "https://dolarapi.com/v1/dolares/blue"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as exc:
        return f"No se pudo consultar DolarAPI: {exc}"

    return f"Dolar blue: compra={data.get('compra')}, venta={data.get('venta')}, actualizado={data.get('fechaActualizacion')}"

print(consultar_dolar_blue.invoke({}))

In [ ]:
llm_dolar = llm.bind_tools([consultar_dolar_blue])
respuesta_dolar = llm_dolar.invoke("Necesito la cotizacion actual del dolar blue para un reporte.")
print(respuesta_dolar)
print("Tool calls:", respuesta_dolar.tool_calls)

for call in respuesta_dolar.tool_calls:
    if call["name"] == "consultar_dolar_blue":
        print("Observacion:", consultar_dolar_blue.invoke(call.get("args", {})))

## Errores comunes

| Error | Explicacion |
|---|---|
| `content=None` | El modelo pidio ejecutar una tool |
| La tool no se ejecuta sola | `tool_calls` es una solicitud |
| La API falla | Manejar timeout y status code |

In [ ]:
assert llm_con_herramientas is not None
assert consultar_dolar_blue.name == "consultar_dolar_blue"
print("Checks OK")

## Resumen

Una tool conecta al modelo con capacidades externas. El docstring y el schema son parte del contrato.